# Mixture of Experts (MoE) - Step-by-Step Explanation, Equations, and PyTorch Code

This notebook explains a sparse Mixture of Experts layer from first principles and then builds a PyTorch implementation that matches the code in this folder.

## What you will learn
- what an expert is
- what the router does
- how top-k routing works
- how expert outputs are combined
- why load-balancing loss is useful
- how MoE replaces the FFN inside a transformer block

## 1. Intuition

A standard transformer feed-forward network sends every token through the same MLP. In MoE, we keep several MLPs, called experts, and let a router decide which experts should process each token.

If there are $E$ experts and each token uses only the best $k$ experts, then computation is sparse: we have many available parameters, but only a small subset is active for each token.

That gives MoE two useful properties:
- larger total model capacity
- lower compute per token than sending the token through every expert

## 2. Core equations

Let the input token representation be $x_{b,t} \in \mathbb{R}^{d}$ for batch index $b$ and token position $t$.

### Step 1: Router logits
The router produces one score per expert:

$$g_{b,t} = W_r x_{b,t} \in \mathbb{R}^{E}$$

where $E$ is the number of experts.

### Step 2: Top-k expert selection
Pick the indices of the $k$ largest router logits:

$$(e_{b,t,1}, \ldots, e_{b,t,k}) = \operatorname{TopK}(g_{b,t})$$

Let the selected logits be $(\ell_{b,t,1}, \ldots, \ell_{b,t,k})$. Convert them to routing weights with a softmax:

$$\alpha_{b,t,i} = \frac{\exp(\ell_{b,t,i})}{\sum_{j=1}^{k} \exp(\ell_{b,t,j})}$$

### Step 3: Expert outputs
Each selected expert is an MLP:

$$y_{b,t,i} = \operatorname{Expert}_{e_{b,t,i}}(x_{b,t}) \in \mathbb{R}^{d}$$

### Step 4: Weighted combination
The final MoE output for that token is the weighted sum of the selected expert outputs:

$$\operatorname{MoE}(x_{b,t}) = \sum_{i=1}^{k} \alpha_{b,t,i} y_{b,t,i}$$

Only the top-k experts are active for the token. That is the sparse part of sparse MoE.

## 3. Tensor shapes
We will use the same conventions as the Python script:
- input `x`: `(B, T, D)`
- router logits: `(B, T, E)`
- top-k indices: `(B, T, k)`
- top-k weights: `(B, T, k)`
- MoE output: `(B, T, D)`

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
torch.set_printoptions(precision=4, sci_mode=False)

## 4. Build the first two pieces: experts and router

We start with two components:
- `ExpertMLP`: one expert network
- `TopKRouter`: the router that chooses the best experts for each token

The router does not run the experts itself. It only decides which experts should receive each token and what weight each selected expert should get.

In [ ]:
class ExpertMLP(nn.Module):
    def __init__(self, d_model: int, d_hidden: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_hidden, d_model)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class TopKRouter(nn.Module):
    def __init__(self, d_model: int, num_experts: int, k: int = 2):
        super().__init__()
        if k > num_experts:
            raise ValueError("k must be <= num_experts")
        self.num_experts = num_experts
        self.k = k
        self.gate = nn.Linear(d_model, num_experts, bias=False)

    def forward(self, x: torch.Tensor):
        router_logits = self.gate(x)
        topk_logits, topk_idx = torch.topk(router_logits, k=self.k, dim=-1)
        topk_weight = F.softmax(topk_logits, dim=-1)
        return topk_idx, topk_weight, router_logits

## 5. Inspect the router on a toy batch

This cell shows exactly what the router produces for each token:
- raw logits over experts
- top-k expert indices
- normalized top-k weights

The top-k weights should sum to 1 for every token because they are produced by a softmax over the selected logits.

In [ ]:
B, T, D = 2, 3, 4
num_experts = 5
k = 2
x = torch.randn(B, T, D)

router = TopKRouter(d_model=D, num_experts=num_experts, k=k)
topk_idx, topk_weight, router_logits = router(x)

print("router_logits shape:", router_logits.shape)
print(router_logits)
print()
print("topk_idx shape:", topk_idx.shape)
print(topk_idx)
print()
print("topk_weight shape:", topk_weight.shape)
print(topk_weight)
print()
print("sum of top-k weights per token:")
print(topk_weight.sum(dim=-1))

## 6. Build the sparse MoE layer

Now we connect the router to the experts.

### Forward-pass idea
1. flatten `(B, T, D)` into `(B*T, D)` so token dispatch is easier
2. look at each expert one by one
3. collect the tokens that selected that expert
4. run only those tokens through that expert
5. multiply each expert output by its routing weight
6. accumulate the weighted results back into the output tensor

This is the key sparse dispatch pattern.

In [ ]:
class MoELayer(nn.Module):
    def __init__(
        self,
        d_model: int,
        d_hidden: int,
        num_experts: int,
        k: int = 2,
        dropout: float = 0.1
    ):
        super().__init__()
        self.d_model = d_model
        self.num_experts = num_experts
        self.k = k

        self.router = TopKRouter(d_model, num_experts, k)
        self.experts = nn.ModuleList([
            ExpertMLP(d_model, d_hidden, dropout=dropout)
            for _ in range(num_experts)
        ])

    def forward(self, x: torch.Tensor):
        B, T, D = x.shape
        if D != self.d_model:
            raise ValueError("last dimension must equal d_model")

        topk_idx, topk_weight, router_logits = self.router(x)

        x_flat = x.reshape(B * T, D)
        topk_idx_flat = topk_idx.reshape(B * T, self.k)
        topk_weight_flat = topk_weight.reshape(B * T, self.k)

        out_flat = torch.zeros_like(x_flat)

        for expert_id, expert in enumerate(self.experts):
            token_pos, kth = torch.where(topk_idx_flat == expert_id)

            if token_pos.numel() == 0:
                continue

            expert_input = x_flat[token_pos]
            expert_output = expert(expert_input)
            weights = topk_weight_flat[token_pos, kth].unsqueeze(-1)
            out_flat[token_pos] += expert_output * weights

        out = out_flat.reshape(B, T, D)
        aux = {
            "router_logits": router_logits,
            "topk_idx": topk_idx,
            "topk_weight": topk_weight
        }
        return out, aux

In [ ]:
B, T, D = 2, 4, 8
d_hidden = 16
num_experts = 4
k = 2
x = torch.randn(B, T, D)

moe = MoELayer(d_model=D, d_hidden=d_hidden, num_experts=num_experts, k=k, dropout=0.0)
out, aux = moe(x)

print("input shape:", x.shape)
print("output shape:", out.shape)
print()
print("top-k expert indices:")
print(aux["topk_idx"])
print()
print("top-k expert weights:")
print(aux["topk_weight"])

## 7. Load-balancing loss

A common failure mode is router collapse: a few experts receive most tokens while others are almost never used. Then model capacity is wasted.

A simple fix is to encourage both of these distributions to be close to uniform:
- `importance`: the soft routing probability mass per expert
- `load`: the actual top-k selection frequency per expert

In this notebook we use:

$$L_{balance} = \operatorname{MSE}(\text{importance}, u) + \operatorname{MSE}(\text{load}, u)$$

where $u$ is the uniform distribution over experts.

In [ ]:
def load_balance_loss(router_logits: torch.Tensor, topk_idx: torch.Tensor, num_experts: int):
    probs = F.softmax(router_logits, dim=-1)

    importance = probs.sum(dim=(0, 1))
    importance = importance / importance.sum()

    one_hot = F.one_hot(topk_idx, num_classes=num_experts).float()
    load = one_hot.sum(dim=(0, 1, 2))
    load = load / load.sum()

    uniform = torch.full_like(load, 1.0 / num_experts)
    return F.mse_loss(importance, uniform) + F.mse_loss(load, uniform)


balance = load_balance_loss(aux["router_logits"], aux["topk_idx"], num_experts=num_experts)
print("load balance loss:", balance.item())

## 8. Put MoE inside a transformer block

In a standard transformer block, self-attention is followed by a dense FFN. Here we replace that FFN with MoE.

The block structure is:
1. layer norm
2. multi-head self-attention
3. residual connection
4. layer norm
5. MoE layer
6. residual connection

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        if d_model % n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads")

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor):
        B, T, D = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        attn_scores = (q @ k.transpose(-2, -1)) / (self.d_head ** 0.5)
        attn_probs = F.softmax(attn_scores, dim=-1)
        attn_probs = self.dropout(attn_probs)

        out = attn_probs @ v
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.out_proj(out)


class TransformerBlockWithMoE(nn.Module):
    def __init__(
        self,
        d_model: int,
        n_heads: int,
        d_hidden: int,
        num_experts: int,
        k: int = 2,
        dropout: float = 0.1
    ):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads, dropout)

        self.ln2 = nn.LayerNorm(d_model)
        self.moe = MoELayer(
            d_model=d_model,
            d_hidden=d_hidden,
            num_experts=num_experts,
            k=k,
            dropout=dropout
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor):
        x = x + self.dropout(self.attn(self.ln1(x)))
        moe_out, aux = self.moe(self.ln2(x))
        x = x + self.dropout(moe_out)
        return x, aux

## 9. End-to-end forward pass and backward pass

This final example matches the script: build the transformer block, run a forward pass, add the auxiliary balancing loss, and backpropagate.

In [ ]:
B, T, D = 4, 16, 64
x = torch.randn(B, T, D)

model = TransformerBlockWithMoE(
    d_model=64,
    n_heads=4,
    d_hidden=128,
    num_experts=6,
    k=2,
    dropout=0.1
)

out, aux = model(x)
aux_loss = load_balance_loss(aux["router_logits"], aux["topk_idx"], num_experts=6)
target = torch.randn_like(out)
task_loss = F.mse_loss(out, target)
total_loss = task_loss + 0.01 * aux_loss
total_loss.backward()

print("input shape:", x.shape)
print("output shape:", out.shape)
print("top-k indices shape:", aux["topk_idx"].shape)
print("top-k weights shape:", aux["topk_weight"].shape)
print("load balance loss:", aux_loss.item())
print("task loss:", task_loss.item())
print("total loss:", total_loss.item())
print("backward success")

## 10. Summary

The MoE pipeline is:
1. compute router logits for each token
2. keep only the top-k experts
3. normalize the selected logits into routing weights
4. run the token through only those experts
5. combine the expert outputs with the routing weights
6. add a load-balancing loss so experts are used more evenly

This notebook gives the basic sparse MoE mechanism. Production systems often add capacity limits, expert parallelism, token dropping, or more advanced balancing objectives, but the core routing idea is the same.